# TP Méthodes d'Ensemble - Partie 2 : Bagging & Random Forest

Comprendre l'impact du Bagging sur la variance et implémenter une version simplifiée de Random Forest.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.utils import resample
from collections import Counter

## 0. Chargement des données

Prérequis : voir `README.md`.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(url, names=columns, na_values='?')
df = df.dropna()
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1).values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset chargé : {X_train.shape[0]} train, {X_test.shape[0]} test")

## 1. Bagging "From Scratch"

1. Générer $B$ échantillons bootstrap (tirage avec remise).  
2. Entraîner un modèle de base (ex: Decision Tree) sur chaque échantillon.  
3. Prédiction : vote majoritaire des $B$ modèles.

In [ ]:
def my_bagging_predict(X_train, y_train, X_test, n_estimators=50, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    models = []
    n_samples = X_train.shape[0]
    
    for i in range(n_estimators):
        # 1. Bootstrap : tirage avec remise de n_samples indices
        idx = resample(np.arange(n_samples), n_samples=n_samples, replace=True, random_state=random_state + i if random_state is not None else None)
        X_boot = X_train[idx]
        y_boot = y_train[idx]
        
        # 2. Entraîner un arbre sur l'échantillon bootstrap
        tree = DecisionTreeClassifier(random_state=random_state)
        tree.fit(X_boot, y_boot)
        models.append(tree)
    
    # 3. Prédiction : vote majoritaire pour chaque point de test
    preds = np.array([m.predict(X_test) for m in models])  # shape (n_estimators, n_test)
    y_pred = np.array([Counter(preds[:, j]).most_common(1)[0][0] for j in range(X_test.shape[0])])
    return y_pred

In [ ]:
# Comparaison : 1 arbre vs Bagging (50 arbres)
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, single_tree.predict(X_test))

y_pred_bag = my_bagging_predict(X_train, y_train, X_test, n_estimators=50, random_state=42)
acc_bag = accuracy_score(y_test, y_pred_bag)

print(f"Arbre unique     : {acc_tree:.4f}")
print(f"Bagging (50 arbres) : {acc_bag:.4f}")

## 2. Bagging avec Sklearn

Vérification avec `BaggingClassifier`.

In [ ]:
bag_sklearn = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    random_state=42
)
bag_sklearn.fit(X_train, y_train)
acc_sklearn = accuracy_score(y_test, bag_sklearn.predict(X_test))
print(f"BaggingClassifier (50 arbres) : {acc_sklearn:.4f}")

## 3. Random Forest

`RandomForestClassifier` ajoute le *feature sampling* (sous-ensemble de features à chaque split) pour décorréler les arbres et souvent améliorer les performances.

In [ ]:
rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
print(f"RandomForestClassifier (50 arbres) : {acc_rf:.4f}")

## 4. (Bonus) Random Forest "From Scratch"

**SimpleDecisionTree** : arbre simplifié avec feature sampling.  
**LightRandomForest** : Bagging + agrégation par vote majoritaire.

In [ ]:
class SimpleDecisionTree:
    def __init__(self, max_depth=5, feature_subsample=None):
        self.max_depth = max_depth
        self.feature_subsample = feature_subsample
        self.tree = None

    def fit(self, X, y):
        self.tree = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        # Arrêt : nœud pur, plus d'échantillons, ou profondeur max
        if depth >= self.max_depth or n_samples == 0 or len(np.unique(y)) == 1:
            return Counter(y).most_common(1)[0][0]

        # Feature sampling : sous-ensemble aléatoire de features
        k = self.feature_subsample if self.feature_subsample is not None else n_features
        k = min(k, n_features)
        indices = np.random.choice(n_features, size=k, replace=False)

        best_feat = np.random.choice(indices)
        threshold = np.mean(X[:, best_feat])

        left_idx = X[:, best_feat] <= threshold
        right_idx = ~left_idx

        if not any(left_idx) or not any(right_idx):
            return Counter(y).most_common(1)[0][0]

        return {
            'feat': best_feat,
            'threshold': threshold,
            'left': self._grow_tree(X[left_idx], y[left_idx], depth + 1),
            'right': self._grow_tree(X[right_idx], y[right_idx], depth + 1)
        }

    def _predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree
        if x[tree['feat']] <= tree['threshold']:
            return self._predict_one(x, tree['left'])
        return self._predict_one(x, tree['right'])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])

In [ ]:
class LightRandomForest:
    def __init__(self, n_trees=10, max_depth=5, max_features='sqrt', random_state=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        n_samples, n_features = X.shape
        if self.random_state is not None:
            np.random.seed(self.random_state)

        if self.max_features == 'sqrt':
            self.f_size = int(np.sqrt(n_features))
        else:
            self.f_size = n_features

        for i in range(self.n_trees):
            # Bootstrap : indices avec remise
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]

            tree = SimpleDecisionTree(max_depth=self.max_depth, feature_subsample=self.f_size)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.array([Counter(sample_preds).most_common(1)[0][0] for sample_preds in tree_preds.T])

In [ ]:
np.random.seed(42)
lrforest = LightRandomForest(n_trees=50, max_depth=5, max_features='sqrt', random_state=42)
lrforest.fit(X_train, y_train)
acc_lrf = accuracy_score(y_test, lrforest.predict(X_test))
print(f"LightRandomForest (50 arbres, sqrt features) : {acc_lrf:.4f}")